In [ ]:
import os

BASE_DIR = "ABSOLUTE_PATH_TO_THE_PROJECT"
DATA_DIR = os.path.join(BASE_DIR, "data")

LOCAL_DATA_ROOT = os.path.expanduser("~")

TASK_YAML = os.path.join(LOCAL_DATA_ROOT, "dist_bench_local_config", "task", "default.yaml")
CODE_DIR = os.path.join(BASE_DIR, "code")

LOCAL_MODEL_ROOT = os.path.join(LOCAL_DATA_ROOT, ".dist_bench_models")
FC_DIR = os.path.join(LOCAL_MODEL_ROOT, "fc")
UC_DIR = os.path.join(LOCAL_MODEL_ROOT, "uc")
os.makedirs(FC_DIR, exist_ok=True)
os.makedirs(UC_DIR, exist_ok=True)

_fc_register = os.path.join(FC_DIR, "model_register.json")
if not os.path.isfile(_fc_register):
    with open(_fc_register, "w", encoding="utf-8") as f:
        f.write("{}")

import sys

sys.path.append(CODE_DIR)

import json
import math
from collections import defaultdict
from typing import Tuple, Optional, Sequence

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import matplotlib.gridspec as grid_spec

import torch
from torch import Tensor, nn

from tqdm.auto import tqdm

from scipy.stats import chi2
from statsmodels.tsa.stattools import acf


from generator import DataGenerator, ChronoSplittedTsDataset
from config import TSDataConfig, TaskConfig
from omegaconf import DictConfig, OmegaConf

from models.forcast.darts import SimpleDartsModel
from models.forcast.forcast_service import ForcastService
from models.forcast.forcast_base import FCPredictionData
from models.uncertainty.uc_service import UncertaintyService
from utils.calc_torch import calc_residuals

In [ ]:
def load_task_config_from_yaml(path: str) -> DictConfig:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"No Task YAML: {path}\n"
            "Please check dist_bench_local_config/task/default.yaml."
        )
    raw = OmegaConf.load(path)
    if "_target_" in raw:
        del raw["_target_"]
    return DictConfig(OmegaConf.to_container(raw, resolve=True))


TASK_CONFIG = load_task_config_from_yaml(TASK_YAML)


def load_dataset(data_config: TSDataConfig, task_config: TaskConfig = None):
    task_config = task_config or TaskConfig(
        **OmegaConf.to_container(TASK_CONFIG, resolve=True)
    )
    return DataGenerator.get_data(
        data_config=data_config,
        task_config=TASK_CONFIG,
        replace_base_dir=DATA_DIR,
        X_norm_param=None,
        Y_norm_param=None,
        hydro_static_norm_param=None,
    )


def load_forecast_service(
    fc_model=None, model_config=None, data_config=None, task_config=None
):
    if fc_model is None:
        fc_model = SimpleDartsModel
        model_config = dict(
            model="darts-forest", model_params={"lags": 50, "lags_past_covariates": 50}
        )

    task_config = task_config or TASK_CONFIG

    return ForcastService(
        int_fc_model=lambda: fc_model(**model_config),
        task_config=task_config,
        model_config=model_config,
        data_config=data_config,
        persist_dir=FC_DIR,
    )

In [47]:
def get_residuals(forcast_service: ForcastService, data, is_calib: bool = False) -> Tuple[np.ndarray, np.ndarray]:
    data = forcast_service.prepare([data], forcast_service._task_config.alpha)[0]

    if is_calib:
        calib_data = UncertaintyService._map_to_calib_data(data)
        fc_result = forcast_service.predict(
            FCPredictionData(
                ts_id=calib_data.ts_id,
                X_past=calib_data.X_pre_calib,
                Y_past=calib_data.Y_pre_calib,
                X_step=calib_data.X_calib,
                step_offset=calib_data.step_offset,
            )
        )
        return calc_residuals(Y_hat=fc_result.point, Y=calib_data.Y_calib).numpy(), fc_result.point

    fc_result = forcast_service.predict(
        FCPredictionData(
            ts_id=data.ts_id,
            X_past=data.X_calib,
            Y_past=data.Y_calib,
            X_step=data.X_test,
            step_offset=data.test_step,
        )
    )
    return calc_residuals(Y_hat=fc_result.point, Y=data.Y_test).numpy(), fc_result.point


def patch_data(
    data: np.ndarray,
    patch_len: int,
) -> np.ndarray:
    return np.lib.stride_tricks.sliding_window_view(data, patch_len)


def get_residuals_dataset(
    forcast_service: ForcastService,
    data: ChronoSplittedTsDataset,
    patch_len: int,
    is_calib: bool = False,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    residuals, preds = get_residuals(forcast_service, data, is_calib)
    residuals = np.ravel(residuals)
    patches = patch_data(residuals, patch_len)
    inputs = patches[:-1]
    targets = patches[1:, -1]
    return inputs, targets, preds[patch_len:]

In [ ]:
from sklearn.cluster import KMeans


def _patch_sequence_to_scalars(series):
    """1D series or (T, w) patch array → per-timestep scalar (median)."""
    a = np.asarray(series)
    if a.ndim >= 2:
        return np.array([np.median(p) for p in a])
    return a.ravel()
 
 
def _patches_to_sorted_vectors(series):
    """
    (T, w) patch array or 1D series → (T, w) sorted quantile vectors.
    Sorted patch = sufficient statistic for the ECDF under KS distance.
    Falls back to column vector for 1D input.
    """
    a = np.asarray(series, dtype=float)
    if a.ndim == 1:
        return a.reshape(-1, 1)
    return np.sort(a, axis=1)
 
 
# ── replacement 1: dCov proxy (drop-in for beta_mixing_acf_proxy) ────────────
 
def _u_center(D: np.ndarray) -> np.ndarray:
    """U-centered distance matrix (Székely & Rizzo 2014)."""
    n = len(D)
    row = D.mean(axis=1, keepdims=True)
    col = D.mean(axis=0, keepdims=True)
    A = D - row - col + D.mean()
    np.fill_diagonal(A, 0.0)
    return A
 
 
def _dcov2_unbiased(X: np.ndarray, Y: np.ndarray) -> float:
    """Unbiased dCov²(X, Y). Returns NaN if n < 4."""
    n = len(X)
    if n < 4:
        return np.nan
    from scipy.spatial.distance import cdist
    A = _u_center(cdist(X, X))
    B = _u_center(cdist(Y, Y))
    return float((A * B).sum() / (n * (n - 3)))
 
 
def beta_mixing_dcov_proxy(patch_sequence, max_lag=10):
    """
    dCov²-based mixing proxy.  Drop-in for beta_mixing_acf_proxy().
 
    Returns
    -------
    dcov2_proxies : ndarray (max_lag,)
        Raw dCov²(Patch_t, Patch_{t+m}) for m = 1..max_lag.
        Clipped to [0, 1] for table parity with the ACF version.
        (True dCov² is non-negative; clipping only matters for rounding artefacts.)
    dcov2_vals : ndarray (max_lag,)
        Same values — second slot kept for API parity with beta_mixing_acf_proxy,
        which returns (beta_proxies, acf_vals).  Callers that ignore the second
        return value are unaffected.
    """
    vecs = _patches_to_sorted_vectors(patch_sequence)
    T = len(vecs)
 
    dcov2_vals = np.full(max_lag, np.nan)
    for m in range(1, max_lag + 1):
        if T - m < 4:
            break
        X = vecs[: T - m]
        Y = vecs[m:]
        dcov2_vals[m - 1] = _dcov2_unbiased(X, Y)
 
    proxies = np.clip(np.nan_to_num(dcov2_vals, nan=np.nan), 0.0, 1.0)
    return proxies, dcov2_vals          # (beta_proxies, acf_vals) slot
 
 
# ── replacement 2: TV-based β̂ block (drop-in for beta_mixing_block) ─────────
 
def beta_mixing_tv_block(series, max_lag, n_bins=10):
    """
    TV-based β̂ estimator.  Drop-in for beta_mixing_block().
 
    Differences from the original:
    - Discretizes patch space via KMeans on sorted quantile vectors
      (consistent with DistMatch's KS framework) instead of percentile bins
      on median scalars.
    - Computes TV(P_joint, P_marginal_product) directly from the contingency
      table — identical formula to the original, but applied to patch-ECDF
      clusters rather than scalar bins.
    - Second return slot carries dCov² (instead of ACF) so that
      print_rebuttal_table_m1's significance annotation is replaced by a
      dCov²-based one.  The significance flag logic in the printer uses
      acf_vals only for display; changing its meaning here is safe.
 
    Returns
    -------
    beta_proxies : ndarray (max_lag,)   — TV-based β̂_m, in [0, 1]
    dcov2_vals   : ndarray (max_lag,)   — dCov² at each lag (second slot)
    """
    vecs = _patches_to_sorted_vectors(series)       # (T, w)
    T = len(vecs)
 
    if T < max_lag + 2:
        return np.full(max_lag, np.nan), np.full(max_lag, np.nan)
 
    # ── cluster entire sequence once for a consistent state space ──────────
    # Rule: n_bins ~ sqrt(T) avoids sparse contingency tables.
    # With T pairs at lag m, the (K×K) joint table has K² cells.
    # We need K² << T, i.e. K << sqrt(T).  Using K = floor(sqrt(T)/2) is
    # conservative; cap at n_bins for large T.
    effective_bins = min(n_bins, max(2, int(np.sqrt(T) / 2)))
    km = KMeans(n_clusters=effective_bins, random_state=42, n_init=10)
    labels = km.fit_predict(vecs).astype(int)       # (T,)  values in [0, K-1]
    K = effective_bins
 
    beta_proxies = np.full(max_lag, np.nan)
    dcov2_vals   = np.full(max_lag, np.nan)
 
    for m in range(1, max_lag + 1):
        n_pairs = T - m
        # TV(β̂) is computable when there are ≥2 lag pairs. (T=4, m=1 → 3 pairs; the existing n≥4 condition is what causes the '—'.)
        # dCov² slot: _dcov2_unbiased returns NaN when n<4.
        if n_pairs < 2:
            break
 
        x = labels[: T - m]        # X_t
        y = labels[m:]              # X_{t+m}
 
        # ── contingency table → joint distribution ─────────────────────────
        contingency = np.zeros((K, K))
        np.add.at(contingency, (x, y), 1)
 
        row_ok = contingency.sum(axis=1) > 0
        col_ok = contingency.sum(axis=0) > 0
        c = contingency[np.ix_(row_ok, col_ok)]
        if c.size == 0:
            continue
 
        joint      = c / c.sum()
        marginal_x = joint.sum(axis=1, keepdims=True)   # P(X_t)
        marginal_y = joint.sum(axis=0, keepdims=True)   # P(X_{t+m})
        product    = marginal_x @ marginal_y             # P(X_t)P(X_{t+m})
 
        # β̂_m = TV(joint, product)  — lower bound (coarser partition → smaller)
        beta_proxies[m - 1] = 0.5 * np.abs(joint - product).sum()
 
        # dCov² on sorted vectors for the same lag (secondary slot)
        X_vecs = vecs[: T - m]
        Y_vecs = vecs[m:]
        dcov2_vals[m - 1] = _dcov2_unbiased(X_vecs, Y_vecs)
 
    return beta_proxies, dcov2_vals     # (beta_proxies, acf_vals) slot
 

In [ ]:
def test_beta_mixing(
    data: ChronoSplittedTsDataset,
    forecast_service: ForcastService,
    patch_lens: Sequence[int],
    max_lag: int = 10,
    min_patches_per_lag: int = 3,  # relaxed from 10 to 5
    lbd_kwargs: Optional[dict] = None,
    patch_stride: Optional[int] = 100,
):
    """patch_stride: subsampling interval for patch indices. 1=all patches (overlapping).
    If None, stride=patch_len (non-overlapping). If patch_stride > patch_len, clip to patch_len."""
    lbd_kwargs = lbd_kwargs if lbd_kwargs is not None else {}

    n_patch_lens = len(patch_lens)
    beta_matrix  = np.full((n_patch_lens, max_lag), np.nan)
    acf_matrix   = np.full((n_patch_lens, max_lag), np.nan)
    t_actual     = np.zeros(n_patch_lens, dtype=int)

    for wi, patch_len in enumerate(patch_lens):
        calib_xs, calib_ys, calib_points = get_residuals_dataset(
            forecast_service, data, patch_len
        )

        patch_sequence_full = np.array(calib_xs)
        if patch_stride is None:
            step = patch_len
        else:
            req = int(patch_stride)
            step = min(req, patch_len)  # stride > window면 patch_len으로 clip
        patch_seq = patch_sequence_full[::step]
        T = len(patch_seq)
        t_actual[wi] = T

        # at minimum, attempt m=1 if T >= min_patches_per_lag
        effective_lag = min(max_lag, T // min_patches_per_lag)
        effective_lag = max(1, effective_lag) if T >= min_patches_per_lag else 0

        if effective_lag == 0:
            print(f"patch_len={patch_len}: subsample_stride={step}, T={T} patches "
                  f"— insufficient (need >={min_patches_per_lag}), reporting NaN.")
            continue

        sub_mode = "non-overlapping subsample" if step == patch_len else f"stride={step}"
        clip_note = ""
        if patch_stride is not None and int(patch_stride) > patch_len:
            clip_note = f" [stride {int(patch_stride)}→{step} clipped to patch_len]"
        print(f"patch_len={patch_len}: patch_subsample {sub_mode}{clip_note}, "
              f"T={T}, effective_lag={effective_lag}")
        #beta_mixing_dcov_proxy, beta_mixing_tv_block
        beta_proxies, acf_vals = beta_mixing_tv_block(
            patch_seq, max_lag=effective_lag
        )

        beta_matrix[wi, :effective_lag] = beta_proxies
        acf_matrix[wi,  :effective_lag] = acf_vals

    return beta_matrix, acf_matrix, t_actual


_RELIABLE_T_THRESHOLD = 30
 
 
def print_rebuttal_table_m1(results, patch_lens, report_lag: int = 1):
    """
    report_lag: the lag m of β̂_m to use in the table (1-indexed). If m=100, set report_lag=100.
    Accordingly, test_beta_mixing(..., max_lag>=100), and T (number of patches) must be
    large enough for effective_lag to reach 100 (roughly T // min_patches_per_lag >= 100).
    """
    dataset_names = list(results.keys())
    lag_idx = report_lag - 1
 
    print(f"\n{'='*65}")
    print(f"Table: β̂_{report_lag} (lag-{report_lag} TV-based estimate) across patch sizes")
    print(f"* = unreliable (T < {_RELIABLE_T_THRESHOLD}, sparse contingency table)")
    print(f"— = insufficient patches")
    print(f"{'='*65}")
 
    print(f"{'w':>8}", end="")
    for dname in dataset_names:
        print(f"  {dname:>10}", end="")
    print()
    print("-" * (8 + 12 * len(dataset_names)))
 
    for wi, w in enumerate(patch_lens):
        print(f"w={w:<6}", end="")
        for dname in dataset_names:
            betas, _, t_vals = results[dname]
            T = t_vals[wi]
            if lag_idx < 0 or lag_idx >= betas.shape[1]:
                b = np.nan
            else:
                b = betas[wi, lag_idx]
 
            if np.isnan(b):
                print(f"  {'—':>10}", end="")
            else:
                flag = "*" if T < _RELIABLE_T_THRESHOLD else " "
                print(f"  {b:.3f}{flag:>1} (T={T:>4})", end="")
        print()
 
    print(f"\nNote: estimates with T < {_RELIABLE_T_THRESHOLD} should be interpreted")
    print(f"cautiously; report w=5,10,25 as primary evidence.")

In [ ]:
PATCH_LENS = [5, 10, 25, 50, 100]
DATA_MAP = {
    # "Elec": (
    #     "electric",
    #     ["/some_base_dir/data/enbPI/electricity-normalized.csv"],
    #     0,
    # ),
    # "Solar": (
    #     "solar",
    #     ["/some_base_dir/data/enbPI/Solar_Atl_data_aligned.csv"],
    #     0,
    # ),
    # "Wind": (
    #     "wind",
    #     ["/some_base_dir/data/enbPI/Wind_Hackberry_Generation_2019_2020.csv"],
    #     0,
    # ),
    # "META": (
    #     "stock",
    #     [os.path.join(LOCAL_DATA_ROOT, "META_5m.csv")],
    #     0,
    # ),
    "NVDA": (
        "stock",
        [os.path.join(LOCAL_DATA_ROOT, "NVDA_5m.csv")],
        0,
    ),
}

results = {}
for data_type, (datatype, data_paths, file_idx) in DATA_MAP.items():
    data_config = DictConfig(
        {"dataset_type": datatype, "paths": data_paths, "add_config": None}
    )
    datasets = load_dataset(TSDataConfig(**data_config))
    forcast_service = load_forecast_service(data_config=data_config)

    beta, acf_mat, t_vals = test_beta_mixing(
        datasets[file_idx],
        forcast_service,
        PATCH_LENS,
        # Up to β̂_100: max_lag=100. effective_lag=min(max_lag, T//min_patches_per_lag) → make T large enough (stride=1, calibration, etc.).
        max_lag=10,
    )
    results[data_type] = (beta, acf_mat, t_vals)

# β̂_1 init. β̂_100: print_rebuttal_table_m1(results, PATCH_LENS, report_lag=100)
print_rebuttal_table_m1(results, PATCH_LENS, report_lag=8)

patch_len=5: patch_subsample non-overlapping subsample [stride 100→5 clipped to patch_len], T=488, effective_lag=10
patch_len=10: patch_subsample non-overlapping subsample [stride 100→10 clipped to patch_len], T=244, effective_lag=10
patch_len=25: patch_subsample non-overlapping subsample [stride 100→25 clipped to patch_len], T=97, effective_lag=10
patch_len=50: patch_subsample non-overlapping subsample [stride 100→50 clipped to patch_len], T=48, effective_lag=10
patch_len=100: patch_subsample non-overlapping subsample, T=24, effective_lag=8

Table: β̂_8 (lag-8 TV-based estimate) across patch sizes
* = unreliable (T < 30, sparse contingency table)
— = insufficient patches
       w        NVDA
--------------------
w=5       0.280  (T= 488)
w=10      0.286  (T= 244)
w=25      0.246  (T=  97)
w=50      0.144  (T=  48)
w=100     0.062* (T=  24)

Note: estimates with T < 30 should be interpreted
cautiously; report w=5,10,25 as primary evidence.
